In [ ]:
# CENTRALITY PIPELINE — Abortion Topic Analysis (2005–2025)
# Models loaded from local paths (no network calls)

import re
import json
import os
from pathlib import Path
import pandas as pd
import torch
from transformers import pipeline
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from openpyxl import load_workbook

# Force transformers to never attempt a network call
os.environ["HF_HUB_OFFLINE"]      = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────

INPUT_CSV        = "output.csv"
OUTPUT_DIR       = Path("./output_centrality")
TOPIC            = "abortion"

ID_COLUMN        = "object_id"
DATE_COLUMN      = "start_date"
TITLE_COLUMN     = "title"
TEXT_COLUMN      = "text"
NEWSPAPER_COLUMN = "_file"

START_DATE            = "2005-01-01"
END_DATE              = "2025-12-31"
SAMPLE_SIZE_PER_MONTH = 100
RANDOM_STATE          = 42
MAX_CHARS             = 2500

# Local path to zero-shot model uploaded to TDM Studio
ZERO_SHOT_MODEL = "./models/deberta-zeroshot"

CLASSIFY_BATCH_SIZE = 8    # lower to 4 if you hit memory issues

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ─── HELPER FUNCTIONS ────────────────────────────────────────

def extract_p_content(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"<p>(.*?)</p>", text, flags=re.DOTALL | re.IGNORECASE)

def strip_html(text):
    return re.sub(r"<[^>]+>", " ", text)

def prepare_text(text):
    if isinstance(text, list):
        text = " ".join(map(str, text))
    text = str(text) if text is not None else ""
    text = strip_html(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def candidate_labels(topic):
    return [
        f"The topic of {topic} is the main focus of this article.",
        f"The topic of {topic} is a major topic in this article.",
        f"The topic of {topic} is one of several important topics in this article.",
        f"The topic of {topic} is mentioned only briefly or as background in this article.",
        f"The topic of {topic} is not relevant to this article.",
    ]

def ordinal_mapping(labels):
    return {
        labels[0]: 5,
        labels[1]: 4,
        labels[2]: 3,
        labels[3]: 2,
        labels[4]: 1,
    }

def continuous_from_ordinal_probs(labels, scores, label_to_ordinal):
    """Converts ordinal probability distribution to a continuous score in [0, 1]."""
    ordinal_to_unit = {1: 0.00, 2: 0.25, 3: 0.50, 4: 0.75, 5: 1.00}
    weighted_score = 0.0
    for lbl, prob in zip(labels, scores):
        weighted_score += ordinal_to_unit[label_to_ordinal[lbl]] * float(prob)
    return round(weighted_score, 4)

def sample_month(group, n=100, random_state=42):
    return group.sample(n=min(len(group), n), random_state=random_state)


In [ ]:
# ─── ANALYZER CLASS ──────────────────────────────────────────

class RefinedCentralityAnalyzer:
    def __init__(self, zero_shot_model_path):
        device_int = 0 if torch.cuda.is_available() else -1

        import os
        print(f"Working directory: {os.getcwd()}")
        resolved = Path(zero_shot_model_path).resolve()
        print(f"  Looking for zero-shot model at: {resolved}")
        if not resolved.exists():
            raise FileNotFoundError(
                f"Zero-shot model folder not found: '{resolved}'\n"
                f"Update ZERO_SHOT_MODEL in CONFIG to match the actual path on TDM Studio."
            )

        print(f"Loading zero-shot model from: {zero_shot_model_path}")
        self.classifier = pipeline(
            "zero-shot-classification",
            model=zero_shot_model_path,
            device=device_int,
            local_files_only=True,
        )
        print("  zero-shot model loaded OK")

    def score_batch(self, texts, topic=TOPIC, max_chars=MAX_CHARS,
                    batch_size=CLASSIFY_BATCH_SIZE):
        labels   = candidate_labels(topic)
        ord_map  = ordinal_mapping(labels)
        snippets = [t[:max_chars] for t in texts]

        scored_rows = []
        for start in range(0, len(snippets), batch_size):
            batch   = snippets[start : start + batch_size]
            results = self.classifier(batch, labels, multi_label=False)
            if isinstance(results, dict):
                results = [results]
            for result in results:
                best_label = result["labels"][0]
                best_score = round(float(result["scores"][0]), 4)
                continuous = continuous_from_ordinal_probs(
                    result["labels"], result["scores"], ord_map
                )
                scored_rows.append({
                    "centrality_ordinal":    ord_map[best_label],
                    "centrality_confidence": best_score,
                    "centrality_continuous": continuous,
                })
        return scored_rows

In [ ]:
# ─── LOAD & VALIDATE DATA ────────────────────────────────────

df_raw = pd.read_csv(INPUT_CSV)
print("Raw shape:", df_raw.shape)
print("Columns:  ", df_raw.columns.tolist())

required_cols = [ID_COLUMN, DATE_COLUMN, TITLE_COLUMN, TEXT_COLUMN, NEWSPAPER_COLUMN]
missing_cols  = [c for c in required_cols if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df_raw[required_cols].copy()


# ─── CLEAN ───────────────────────────────────────────────────

df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
df = df.dropna(subset=[DATE_COLUMN])
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").apply(extract_p_content).apply(prepare_text)

print("Cleaned shape:", df.shape)


# ─── FILTER: TOPIC + DATE RANGE ──────────────────────────────

df_topic = df[
    df[TEXT_COLUMN].str.contains(r"\babortion\b", case=False, na=False, regex=True)
].copy()
print("Rows containing abortion:", len(df_topic))

df_topic = df_topic[
    (df_topic[DATE_COLUMN] >= pd.to_datetime(START_DATE)) &
    (df_topic[DATE_COLUMN] <= pd.to_datetime(END_DATE))
].copy()
df_topic["year_month"] = df_topic[DATE_COLUMN].dt.to_period("M").astype(str)
print("After date filtering:", df_topic.shape)


# ─── SAMPLE: MAX 100 PER MONTH ───────────────────────────────

sampled_df = (
    df_topic
    .groupby("year_month", group_keys=False)
    .apply(lambda g: sample_month(g, n=SAMPLE_SIZE_PER_MONTH, random_state=RANDOM_STATE))
    .reset_index(drop=True)
)
# Drop any stale index columns pandas may inject (level_0, index)
sampled_df = sampled_df.loc[:, ~sampled_df.columns.str.match(r"^(level_|index$)")]
print("Sampled shape:", sampled_df.shape)

In [ ]:
# ─── APPLY CENTRALITY PIPELINE ───────────────────────────────

analyzer = RefinedCentralityAnalyzer(
    zero_shot_model_path=ZERO_SHOT_MODEL,
)

# Drop empty-text rows
sampled_df = sampled_df[
    sampled_df[TEXT_COLUMN].str.strip().astype(bool)
].reset_index(drop=True)

texts = sampled_df[TEXT_COLUMN].tolist()

print(f"\nRunning zero-shot classification on {len(texts)} articles...")
scored_rows = analyzer.score_batch(texts)

records = [
    {
        "article_id":  sampled_df.at[i, ID_COLUMN],
        "article_date": sampled_df.at[i, DATE_COLUMN].strftime("%Y-%m-%d"),
        "year_month":  sampled_df.at[i, "year_month"],
        "title":       sampled_df.at[i, TITLE_COLUMN],
        "newspaper":   sampled_df.at[i, NEWSPAPER_COLUMN],
        **scored_rows[i],
    }
    for i in range(len(sampled_df))
]

results_df = pd.DataFrame(records)
print(f"Done — {len(records)} articles processed.")

In [ ]:
# ─── OUTPUT 1: JSON ───────────────────────────────────────────
# Columns: article_id, article_date, title, word_embedding, abortion

json_records = [
    {
        "article_id":   r["article_id"],
        "article_date": r["article_date"],
        "title":        r["title"],
        "abortion":     r["centrality_continuous"],
    }
    for r in records
]

json_path = OUTPUT_DIR / "article_centrality.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_records, f, ensure_ascii=False, indent=2)
print("Saved:", json_path)

In [ ]:
# ─── SUMMARY TABLE ───────────────────────────────────────────

summary = (
    results_df
    .groupby("year_month")
    .agg(
        n_articles           = (ID_COLUMN,               "count"),
        mean_centrality_cont = ("centrality_continuous", "mean"),
        mean_centrality_ord  = ("centrality_ordinal",    "mean"),
        mean_confidence      = ("centrality_confidence", "mean"),
        pct_high_centrality  = ("centrality_ordinal",
                                lambda x: round((x >= 4).mean() * 100, 1)),
    )
    .reset_index()
)
summary["year_month_dt"] = pd.to_datetime(summary["year_month"])
summary = summary.sort_values("year_month_dt").reset_index(drop=True)

print("\nMonthly summary (first 5 rows):")
print(summary.head())

In [ ]:
# ─── OUTPUT 2: EXCEL — summary table + chart ─────────────────

excel_path = OUTPUT_DIR / "centrality_summary_report.xlsx"

export_summary = summary.drop(columns=["year_month_dt"]).rename(columns={
    "year_month":           "Year-Month",
    "n_articles":           "N Articles",
    "mean_centrality_cont": "Mean Centrality (0-1)",
    "mean_centrality_ord":  "Mean Centrality (Ordinal)",
    "mean_confidence":      "Mean Confidence",
    "pct_high_centrality":  "% High Centrality (Ord >= 4)",
})

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    export_summary.to_excel(writer, sheet_name="Monthly Summary", index=False)

wb = load_workbook(excel_path)
ws = wb["Monthly Summary"]

for col in ws.columns:
    max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
    ws.column_dimensions[col[0].column_letter].width = max(max_len + 2, 14)

wb.save(excel_path)
print("Saved:", excel_path)